# 03 - Transform Tanqeeb Job Data

This notebook is the **transformation** stage of the Tanqeeb pipeline.

- Reads ONLY the jobs waiting in the *pending cleaned queue*
  (`tanqeeb_pending_cleaned_queue.json`) -- i.e. jobs `02_clean_Tanqeeb.ipynb`
  has cleaned but that haven't been transformed yet.
- Enforces the final column order (`FINAL_SCHEMA`) on just that batch:
  `job_id`, `source`, `job_title`, `company_name`, `job_description`,
  `job_url`, `country`, `city`, `work_mode`, `employment_type`,
  `career_level`, `experience_needed`, `qualification_required`, `skills`,
  `salary`, `estimated_salary`, `posted_date`, `collected_at`.
- **Appends** the transformed batch onto the historical final dataset
  (CSV + JSON), instead of rebuilding it from scratch, then clears the
  cleaned queue it just consumed.


In [1]:
import json
import os
import pandas as pd


In [2]:
# ============================================================
# CONFIG
# ============================================================

# Input: produced (appended to) by 02_clean_Tanqeeb.ipynb
CLEANED_PENDING_QUEUE = "../data/interim/tanqeeb_pending_cleaned_queue.json"

# Cumulative, historical final dataset -- this notebook APPENDS to it
FINAL_JSON_OUTPUT = "../data/processed/tanqeeb_jobs_final.json"
FINAL_CSV_OUTPUT = "../data/processed/tanqeeb_jobs_final.csv"

# Final standardized column order
FINAL_SCHEMA = [
    "job_id",
    "source",
    "job_title",
    "company_name",
    "job_description",
    "job_url",
    "country",
    "city",
    "work_mode",
    "employment_type",
    "career_level",
    "experience_needed",
    "qualification_required",
    "skills",
    "salary",
    "estimated_salary",
    "posted_date",
    "collected_at",
]


In [3]:
def load_json_list(path):
    """Load a JSON file containing a list of job dicts; [] if missing."""
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_json_list(path, items):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)


## Load ONLY the new (not-yet-transformed) jobs from the pending queue

In [4]:
cleaned_jobs = load_json_list(CLEANED_PENDING_QUEUE)
print("New cleaned jobs waiting to be transformed:", len(cleaned_jobs))

df = pd.DataFrame(cleaned_jobs)
df.head()


New cleaned jobs waiting to be transformed: 0


""


## Transform: enforce final schema (add missing columns, reorder)

In [5]:
# Ensure every standardized column exists, even if a field was
# never populated for any job (keeps output schema consistent).
for col in FINAL_SCHEMA:
    if col not in df.columns:
        df[col] = "N/A"

df = df[FINAL_SCHEMA]


## Append this batch to the historical final dataset

In [6]:
transformed_batch = df.to_dict(orient="records")

history_df = pd.DataFrame(load_json_list(FINAL_JSON_OUTPUT))
combined_df = pd.concat([history_df, df], ignore_index=True)

# Idempotent: if this notebook is accidentally re-run on the same batch,
# duplicates against history are dropped rather than piling up.
if "job_url" in combined_df.columns and not combined_df.empty:
    combined_df = combined_df.drop_duplicates(subset=["job_url"], keep="last")

for col in FINAL_SCHEMA:
    if col not in combined_df.columns:
        combined_df[col] = "N/A"

combined_df = combined_df[FINAL_SCHEMA]

os.makedirs(os.path.dirname(FINAL_CSV_OUTPUT), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_JSON_OUTPUT), exist_ok=True)

combined_df.to_csv(FINAL_CSV_OUTPUT, index=False, encoding="utf-8-sig")
save_json_list(FINAL_JSON_OUTPUT, combined_df.to_dict(orient="records"))


## Clear the cleaned queue we just consumed, and show results

In [7]:
# These cleaned jobs have now been transformed -- remove them from the queue
save_json_list(CLEANED_PENDING_QUEUE, [])

print("\n" + "=" * 60)
print("Jobs transformed this run:", len(transformed_batch))
print("Total jobs in final dataset:", len(combined_df))
print("Final CSV saved as:", FINAL_CSV_OUTPUT)
print("Final JSON saved as:", FINAL_JSON_OUTPUT)
print("Cleaned pending queue cleared.")
print("=" * 60)

combined_df.head()



Jobs transformed this run: 0
Total jobs in final dataset: 10001
Final CSV saved as: ../data/processed/tanqeeb_jobs_final.csv
Final JSON saved as: ../data/processed/tanqeeb_jobs_final.json
Cleaned pending queue cleared.


,job_id,source,job_title,company_name,job_description,job_url,country,city,work_mode,employment_type,career_level,experience_needed,qualification_required,skills,salary,estimated_salary,posted_date,collected_at
0,21225095,Tanqeeb,Front Office Manager required for work in Makkah,N/A,A hotel management in Makkah announces a job v...,https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...,Saudi,Makkah,On-site,Full Time,N/A,2 to 5 years,جامعي (بكالوريوس / ليسانس),N/A,N/A,N/A,10/9/2026 2:47:49 PM,12/9/2026 12:51:34 AM
1,21223553,Tanqeeb,Real Estate Sales Representative Wanted,Imtilak Group,Imtilak Global Real Estate is looking for new ...,https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...,Saudi,Medina,On-site,Full Time,N/A,2 to 4 years,جامعي (بكالوريوس / ليسانس),N/A,N/A,N/A,9/9/2026 5:38:43 PM,12/9/2026 12:51:46 AM
2,21223456,Tanqeeb,Computer Programmer,خبرة لتحصيل الديون,"Designing, developing, and maintaining softwar...",https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...,Saudi,Riyadh,On-site,Full Time,N/A,1 to 6 years,جامعي (بكالوريوس / ليسانس),N/A,N/A,N/A,9/9/2026 2:24:19 PM,12/9/2026 12:51:57 AM
3,21221858,Tanqeeb,Graphic Designer,خبرة لتحصيل الديون,At least one year of experience in the same fi...,https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...,Saudi,Riyadh,On-site,Full Time,N/A,1 to 5 years,جامعي (بكالوريوس / ليسانس),N/A,N/A,N/A,8/9/2026 4:56:35 PM,12/9/2026 12:52:04 AM
4,21221113,Tanqeeb,Sales Manager Required – Hotel Supplies,نجد العالي للتسويق,A specialized marketing company operating in t...,https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...,Saudi,Riyadh,On-site,Full Time,N/A,5 to 10 years,جامعي (بكالوريوس / ليسانس),N/A,N/A,N/A,8/9/2026 2:05:12 PM,12/9/2026 12:52:12 AM
